In [1]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from etl.transformations.common import (
    create_spark,
    write_clickhouse,
    read_batches_since,
)
from etl.transformations.state import (
    get_watermark,
    set_watermark,
)

WATERMARK_PATH = "s3a://staging/_checkpoints/spark/fact_harvests/watermark.json"

SOURCE_PATH = "s3a://staging/raw/postgres/harvests/"

In [2]:
def main():

    spark = create_spark("fact_harvests")

    last_batch = get_watermark(
        spark,
        WATERMARK_PATH,
    )

    # read new batches
    harvests_df, new_batch = read_batches_since(
        spark,
        SOURCE_PATH,
        last_batch,
    )

    harvests_df.printSchema()
    harvests_df.show(5)
    print(new_batch)

    # transform

    # write ClickHouse

    # update watermark

In [3]:
if __name__ == "__main__":
    main()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/22 19:51:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/22 19:51:28 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


SparkRuntimeException: [CONFLICTING_DIRECTORY_STRUCTURES] Conflicting directory structures detected.
Suspicious paths:

	s3a://staging/raw/postgres/harvests/19700101t000000z__20260331t235900z
	s3a://staging/raw/postgres/harvests/20260331t235900z__20260722t172532z

If provided paths are partition directories, please set "basePath" in the options of the data source to specify the root directory of the table.
If there are multiple root directories, please load them separately and then union them. SQLSTATE: KD009